# 01 · Explore a robot recording

**Agenda: 10–25 min.** A DROID teleop episode is one `.mcap` file: three cameras, depth, fused point clouds, 3D cuboids, joint state, gripper state, and an episode summary — on one clock. In FiftyOne ≥ 1.21 a sample whose filepath is an `.mcap` **is** the episode. No conversion, no ROS.

You have 10 episodes locally (`droid-mcap-workshop`). The presenter has 100 on demo.fiftyone.ai. Same workflow.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F

episodes = fo.load_dataset("droid-mcap-workshop")
print(episodes.media_type, len(episodes), "episodes")
episodes

In [ ]:
session = fo.launch_app(episodes)

## In the App

1. **Grid** — every tile is an episode with a playing preview. Use the stream selector to switch the preview camera.
2. **Open an episode** — arrange the tiles: wrist camera, an external camera, the 3D view. Scrub. Everything moves on one clock.
3. **Click a cuboid** in the 3D tile → label, entity id, topic.
4. **Timeline tracks** — expand them. `grasp` / `release` intervals are *temporal tags*: labels on a time range, not a frame.
5. **Sidebar** — filter by `task_type`, `success`, sort by `duration_s`.

### Episodes are queryable like any dataset

In [ ]:
print(episodes.count_values("task_type"))
print(episodes.count_values("success"))

# Suspiciously short "successful" demonstrations
short = episodes.match(F("duration_s") < 4).sort_by("duration_s")
for s in short.select_fields(["episode_id", "duration_s", "success", "current_task", "gripper_open_frac"]):
    print(f"{s.duration_s:5.1f}s  success={s.success!s:5}  gripper_open_frac={s.gripper_open_frac:.2f}  {s.current_task[:60]}")

The 1.4 s episode: the gripper never closed (`gripper_open_frac == 1.0`), marked failed. The 2.8 s one is marked a *success* — worth a look. Open both.

In [ ]:
session.view = short

### Temporal tags: labels on a time range

The dataset ships with `grasp` and `release` intervals. Query them:

In [ ]:
print(episodes.temporal_tags.count())

grasping = episodes.match_temporal_tags(tags="grasp")
print(len(grasping), "episodes contain a grasp interval")
session.view = grasping

**Try it:** open an episode, press **Shift+T** for tag mode, drag an interval on the timeline and name it `arm-stall`. Then:

In [ ]:
# after tagging in the App
print(episodes.temporal_tags.count())
print(len(episodes.match_temporal_tags(tags="arm-stall")), "episodes with an arm-stall")

### What just happened

- One file per episode, browsable and queryable without a conversion pipeline.
- Curation questions ("which demos are degenerate?") are field queries, not scripts.
- Quality lives at the *episode* and *interval* level — the temporal tags are the unit we will come back to in notebook 06.

**Next:** to train a detector we need images. Notebook 02 pulls frames out of these recordings.